# Nhận diện cảm xúc khuôn mặt - Train AffectNet từ Google Drive

Notebook này giữ cùng workflow training với bản Kaggle/local, nhưng đọc dataset từ Google Drive khi chạy trên Google Colab.

## 1. Cài thư viện

Chạy cell này một lần cho mỗi môi trường Colab/runtime nếu các package chưa có sẵn.

In [ ]:
!pip install -q tensorflow opencv-python scikit-learn matplotlib pandas pillow

## 2. Import thư viện và thiết lập đường dẫn project

In [ ]:
# Thư viện cơ bản của Python: xử lý file JSON, hệ điều hành, random, và đường dẫn thư mục.
import json
import os
import random
from pathlib import Path

# Thư viện xử lý ảnh, hiển thị biểu đồ, xử lý mảng/bảng dữ liệu, và train model.
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image

# Các hàm của scikit-learn dùng để chia data, tính class weight, và đánh giá model.
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# Keras là API chính để tạo và train neural network trong TensorFlow.
from tensorflow import keras
from tensorflow.keras import layers

# Lấy thư mục hiện tại mà notebook đang chạy.
# Nếu chạy trong folder notebooks, dòng bên dưới sẽ trỏ tới .../Emotion Detect/notebooks.
NOTEBOOK_DIR = Path.cwd().resolve()

# Tìm thư mục gốc của project.
# Nếu thư mục hiện tại có folder data thì nó chính là project root.
# Nếu không, lấy thư mục cha vì notebook thường nằm trong folder notebooks.
PROJECT_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / 'data').exists() else NOTEBOOK_DIR.parent

# Các đường dẫn chính trong project.
DATA_DIR = PROJECT_ROOT / 'data'                    # Nơi chứa toàn bộ dữ liệu.
RAW_DIR = DATA_DIR / 'raw'                          # Nơi chứa data gốc, chưa xử lý.
PROCESSED_DIR = DATA_DIR / 'processed'              # Nơi chứa data đã xử lý nếu cần.
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'          # Nơi lưu các file sinh ra sau khi train/evaluate.
MODEL_DIR = ARTIFACTS_DIR / 'models'                # Nơi lưu model đã train.
OUTPUT_DIR = ARTIFACTS_DIR / 'outputs'              # Nơi lưu biểu đồ, báo cáo, ảnh output.
UPLOAD_DIR = OUTPUT_DIR / 'uploaded_images'         # Nơi lưu ảnh upload để test dự đoán cảm xúc.

# Tạo các folder cần thiết nếu chúng chưa tồn tại.
# parents=True: tạo luôn folder cha nếu thiếu.
# exist_ok=True: nếu folder đã có sẵn thì không báo lỗi.
for path in [RAW_DIR, PROCESSED_DIR, MODEL_DIR, OUTPUT_DIR, UPLOAD_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# In thông tin để kiểm tra môi trường và đường dẫn đang dùng.
print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))
print('Project root:', PROJECT_ROOT)
print('Raw data dir:', RAW_DIR)
print('Model dir:', MODEL_DIR)

## 3. Cấu hình

Đặt `GOOGLE_DRIVE_DATASET_ROOTS` tới folder Google Drive chứa `labels.csv`, `Train/`, và `Test/`. Trên Colab, Google Drive được mount tại `/content/drive/MyDrive`.

In [ ]:
SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_HEAD = 8
EPOCHS_FINE_TUNE = 8
VALID_SIZE = 0.15
TEST_SIZE = 0.15
LEARNING_RATE_HEAD = 1e-3
LEARNING_RATE_FINE = 1e-5
USE_MIXED_PRECISION = True

# Folder dataset trên Google Drive cho archive này.
# Cấu trúc mong đợi: MyDrive/Emotion Detect/archive/archive (3)/labels.csv
#                  MyDrive/Emotion Detect/archive/archive (3)/Train/...
#                  MyDrive/Emotion Detect/archive/archive (3)/Test/...
GOOGLE_DRIVE_DATASET_ROOTS = [
    '/content/drive/MyDrive/Emotion Detect/archive/archive (3)'
]

# Folder dự phòng khi chạy local thay vì Colab.
LOCAL_DATASET_ROOTS = []

CLASS_NAMES = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
CLASS_TO_ID = {name: idx for idx, name in enumerate(CLASS_NAMES)}
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

AFFECTNET_ID_TO_LABEL = {
    0: 'neutral',
    1: 'happy',
    2: 'sad',
    3: 'surprise',
    4: 'fear',
    5: 'disgust',
    6: 'angry',
    7: None,
    8: None,
    9: None,
    10: None,
}

LABEL_ALIASES = {
    'angry': 'angry', 'anger': 'angry', 'angriness': 'angry',
    'disgust': 'disgust', 'disgusted': 'disgust',
    'fear': 'fear', 'fearful': 'fear',
    'happy': 'happy', 'happiness': 'happy',
    'neutral': 'neutral',
    'sad': 'sad', 'sadness': 'sad',
    'surprise': 'surprise', 'surprised': 'surprise',
}

AFFECTNET_LABEL_COLUMNS = [
    'expression', 'exp', 'emotion', 'label', 'class', 'category', 'emotion_label', 'expression_label'
]
AFFECTNET_PATH_COLUMNS = [
    'pth', 'subDirectory_filePath', 'subdirectory_filepath', 'filepath', 'file_path', 'path', 'image', 'image_path', 'filename', 'file'
]

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

if USE_MIXED_PRECISION and tf.config.list_physical_devices('GPU'):
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print('Mixed precision enabled.')
else:
    print('Mixed precision disabled.')

## 4. Tìm dataset từ Google Drive

In [ ]:
def is_colab_runtime():
    try:
        import google.colab  # type: ignore
        return True
    except ImportError:
        return False


def mount_google_drive():
    if not is_colab_runtime():
        print('Không chạy trên Colab; bỏ qua bước mount Google Drive.')
        return

    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')


def resolve_existing_roots(paths):
    roots = [Path(p).expanduser().resolve() for p in paths]
    existing = [p for p in roots if p.exists()]
    missing = [str(p) for p in roots if not p.exists()]
    if missing:
        print('Không tìm thấy dataset root:', missing)
    return existing


def find_local_raw_roots(raw_dir):
    if not raw_dir.exists():
        return []
    return [p.resolve() for p in raw_dir.iterdir() if p.is_dir()]


def find_google_drive_dataset_roots():
    mount_google_drive()

    drive_roots = resolve_existing_roots(GOOGLE_DRIVE_DATASET_ROOTS)
    if drive_roots:
        print('Đang dùng dataset root từ Google Drive:', drive_roots)
        return drive_roots

    if LOCAL_DATASET_ROOTS:
        local_roots = resolve_existing_roots(LOCAL_DATASET_ROOTS)
        if local_roots:
            print('Đang dùng LOCAL_DATASET_ROOTS:', local_roots)
            return local_roots

    local_raw_roots = find_local_raw_roots(RAW_DIR)
    if local_raw_roots:
        print('Đang dùng các folder dataset đã có trong data/raw:', local_raw_roots)
        return local_raw_roots

    raise FileNotFoundError(
        'Không tìm thấy folder dataset. Hãy cập nhật GOOGLE_DRIVE_DATASET_ROOTS tới folder Drive chứa labels.csv, Train/, và Test/, '
        'ví dụ /content/drive/MyDrive/Emotion Detect/archive/archive (3).'
    )


dataset_roots = find_google_drive_dataset_roots()
dataset_roots

## 5. Chuẩn hóa label và tạo index ảnh

In [ ]:
def normalize_label(value, numeric_mapping='affectnet'):
    if pd.isna(value):
        return None

    text = str(value).strip().lower().replace(' ', '_')
    if text in {'', 'nan', 'none', 'uncertain', 'non_face', 'non-face', 'contempt'}:
        return None

    try:
        numeric = int(float(text))
        if numeric_mapping == 'affectnet':
            return AFFECTNET_ID_TO_LABEL.get(numeric)
    except ValueError:
        pass

    return LABEL_ALIASES.get(text)


def infer_label_from_path(path):
    parts = [p.lower().replace(' ', '_') for p in Path(path).parts]
    for part in reversed(parts):
        label = normalize_label(part, numeric_mapping='affectnet')
        if label is not None:
            return label
    return None


def first_matching_column(columns, candidates):
    normalized = {str(col).strip().lower(): col for col in columns}
    for candidate in candidates:
        if candidate.lower() in normalized:
            return normalized[candidate.lower()]
    return None


def find_existing_image(root, value):
    candidate = Path(str(value).strip())
    candidates = []
    if candidate.is_absolute():
        candidates.append(candidate)
    else:
        clean_value = str(value).strip().lstrip('/\\')
        train_test_candidates = [
            Path(root) / 'Train' / clean_value,
            Path(root) / 'Test' / clean_value,
        ]
        parts = Path(clean_value).parts
        if parts and parts[0].lower() in {'anger', 'contempt'}:
            title_case_value = str(Path(parts[0].title(), *parts[1:]))
            train_test_candidates.append(Path(root) / 'Test' / title_case_value)

        candidates.extend([
            Path(root) / candidate,
            Path(root) / clean_value,
            *train_test_candidates,
            Path(root) / 'Manually_Annotated' / 'Manually_Annotated_Images' / clean_value,
            Path(root) / 'Automatically_Annotated' / 'Automatically_Annotated_Images' / clean_value,
        ])

    for item in candidates:
        if item.exists() and item.suffix.lower() in IMAGE_EXTS:
            return item.resolve()
    return None


def collect_affectnet_csv_records(root):
    records = []
    for csv_path in Path(root).rglob('*.csv'):
        try:
            sample = pd.read_csv(csv_path, nrows=5)
        except Exception:
            continue

        path_col = first_matching_column(sample.columns, AFFECTNET_PATH_COLUMNS)
        label_col = first_matching_column(sample.columns, AFFECTNET_LABEL_COLUMNS)
        if path_col is None or label_col is None:
            continue

        print('Đang đọc annotation AffectNet:', csv_path)
        for chunk in pd.read_csv(csv_path, usecols=[path_col, label_col], chunksize=50000):
            chunk = chunk.dropna(subset=[path_col, label_col])
            for row in chunk.itertuples(index=False):
                image_value, label_value = row
                label = normalize_label(label_value, numeric_mapping='affectnet')
                if label is None:
                    continue
                image_path = find_existing_image(root, image_value)
                if image_path is None:
                    continue
                records.append({
                    'path': str(image_path),
                    'label': label,
                    'source': Path(root).name,
                    'annotation_file': csv_path.name,
                })
    return records


def collect_folder_label_records(root):
    records = []
    for file_path in Path(root).rglob('*'):
        if file_path.suffix.lower() not in IMAGE_EXTS:
            continue
        label = infer_label_from_path(file_path.relative_to(root))
        if label is not None:
            records.append({
                'path': str(file_path.resolve()),
                'label': label,
                'source': Path(root).name,
                'annotation_file': None,
            })
    return records


def collect_image_files(dataset_roots):
    records = []
    for root in dataset_roots:
        root = Path(root)
        csv_records = collect_affectnet_csv_records(root)
        if csv_records:
            records.extend(csv_records)
        else:
            records.extend(collect_folder_label_records(root))

    df_images = pd.DataFrame(records)
    if df_images.empty:
        return pd.DataFrame(columns=['path', 'label', 'source', 'annotation_file'])

    df_images = df_images.drop_duplicates('path').reset_index(drop=True)
    df_images = df_images[df_images['label'].isin(CLASS_NAMES)].copy()
    df_images['label_id'] = df_images['label'].map(CLASS_TO_ID)
    return df_images


df_images = collect_image_files(dataset_roots)
print('Số ảnh tìm thấy:', len(df_images))
display(df_images['label'].value_counts().reindex(CLASS_NAMES).fillna(0).astype(int))
df_images.head()

## 5.1 Xem trước dữ liệu

Chạy các cell này trước khi train để kiểm tra dataset đã tìm được.

In [ ]:
print('Tổng số ảnh:', len(df_images))
print('Số nguồn dữ liệu:', df_images['source'].nunique() if not df_images.empty else 0)
print('Số file bị thiếu:', 0 if df_images.empty else int((~df_images['path'].map(lambda p: Path(p).exists())).sum()))

display(df_images.head(10))

overview = pd.DataFrame({
    'count': df_images['label'].value_counts(),
    'percent': (df_images['label'].value_counts(normalize=True) * 100).round(2),
}).reindex(CLASS_NAMES).fillna(0)
display(overview)

In [ ]:
if df_images.empty:
    print('Không có ảnh để vẽ biểu đồ.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    label_counts = df_images['label'].value_counts().reindex(CLASS_NAMES).fillna(0)
    label_counts.plot(kind='bar', ax=axes[0], color='#4C78A8')
    axes[0].set_title('Số ảnh theo cảm xúc')
    axes[0].set_xlabel('Cảm xúc')
    axes[0].set_ylabel('Số ảnh')
    axes[0].tick_params(axis='x', rotation=35)

    source_counts = df_images['source'].value_counts().head(10)
    source_counts.plot(kind='barh', ax=axes[1], color='#F58518')
    axes[1].set_title('Các nguồn dataset chính')
    axes[1].set_xlabel('Số ảnh')
    axes[1].set_ylabel('Nguồn')

    plt.tight_layout()
    plt.show()

In [ ]:
samples_per_class = 4
sample_df = (
    df_images.groupby('label', group_keys=False)
    .apply(lambda x: x.sample(min(len(x), samples_per_class), random_state=SEED))
    .reset_index(drop=True)
)

if sample_df.empty:
    print('Không có ảnh mẫu để hiển thị.')
else:
    labels_present = [label for label in CLASS_NAMES if label in set(sample_df['label'])]
    rows = len(labels_present)
    cols = samples_per_class
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = np.array(axes).reshape(rows, cols)

    for row_idx, label in enumerate(labels_present):
        label_samples = sample_df[sample_df['label'] == label].reset_index(drop=True)
        for col_idx in range(cols):
            ax = axes[row_idx, col_idx]
            ax.axis('off')
            if col_idx >= len(label_samples):
                continue
            image_path = Path(label_samples.loc[col_idx, 'path'])
            try:
                image = Image.open(image_path).convert('RGB')
                ax.imshow(image)
                ax.set_title(label, fontsize=10)
            except Exception as exc:
                ax.text(0.5, 0.5, f'Không đọc được\n{image_path.name}', ha='center', va='center')
                ax.set_title(label, fontsize=10)

    plt.tight_layout()
    plt.show()

## 6. Chia tập train / validation / test

In [ ]:
if df_images.empty:
    raise ValueError('Không tìm thấy dữ liệu AffectNet hợp lệ. Hãy kiểm tra dataset local hoặc đường dẫn Google Drive.')

min_class_count = int(df_images['label'].value_counts().min())
if min_class_count < 2:
    raise ValueError('Mỗi class cần ít nhất 2 mẫu để chia dữ liệu theo stratify.')

train_df, temp_df = train_test_split(
    df_images,
    test_size=VALID_SIZE + TEST_SIZE,
    stratify=df_images['label'],
    random_state=SEED,
)

relative_test = TEST_SIZE / (VALID_SIZE + TEST_SIZE)
valid_df, test_df = train_test_split(
    temp_df,
    test_size=relative_test,
    stratify=temp_df['label'],
    random_state=SEED,
)

split_summary = pd.DataFrame({
    'train': train_df['label'].value_counts(),
    'valid': valid_df['label'].value_counts(),
    'test': test_df['label'].value_counts(),
}).reindex(CLASS_NAMES).fillna(0).astype(int)

print(len(train_df), len(valid_df), len(test_df))
display(split_summary)

train_df.to_csv(PROCESSED_DIR / 'train_split.csv', index=False)
valid_df.to_csv(PROCESSED_DIR / 'valid_split.csv', index=False)
test_df.to_csv(PROCESSED_DIR / 'test_split.csv', index=False)
print('Đã lưu các file CSV split vào:', PROCESSED_DIR)

## 7. Tạo TensorFlow Dataset

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    return image, label


def make_dataset(dataframe, training=False):
    paths = dataframe['path'].astype(str).values
    labels = dataframe['label_id'].astype(np.int32).values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(min(len(dataframe), 10000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)
    return ds


train_ds = make_dataset(train_df, training=True)
valid_ds = make_dataset(valid_df)
test_ds = make_dataset(test_df)

class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(len(CLASS_NAMES)),
    y=train_df['label_id'].values,
)
class_weights = {i: float(w) for i, w in enumerate(class_weights_arr)}
print('Class weights:', class_weights)

## 8. Build và train model

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.10),
], name='augmentation')

base_model = keras.applications.MobileNetV2(
    include_top=False,
    weights='imagenet',
    input_shape=IMG_SIZE + (3,),
)
base_model.trainable = False

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.35)(x)
outputs = layers.Dense(len(CLASS_NAMES), activation='softmax', dtype='float32')(x)

model = keras.Model(inputs, outputs)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE_HEAD),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

checkpoint_path = MODEL_DIR / 'best_emotion_model.keras'
callbacks = [
    keras.callbacks.ModelCheckpoint(
        checkpoint_path,
        monitor='val_accuracy',
        mode='max',
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        mode='max',
        patience=5,
        restore_best_weights=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=2,
        min_lr=1e-6,
    ),
]

history_head = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=EPOCHS_HEAD,
    callbacks=callbacks,
    class_weight=class_weights,
)

base_model.trainable = True
fine_tune_at = int(len(base_model.layers) * 0.70)
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE_FINE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

history_fine = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=EPOCHS_FINE_TUNE,
    callbacks=callbacks,
    class_weight=class_weights,
)

## 9. Đánh giá và lưu artifact

In [ ]:
def merge_history(*histories):
    merged = {}
    for hist in histories:
        for key, values in hist.history.items():
            merged.setdefault(key, []).extend(values)
    return merged

best_model = keras.models.load_model(checkpoint_path)
test_loss, test_acc = best_model.evaluate(test_ds)
print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {test_acc:.4f}')

y_true = []
y_pred = []
for images, labels in test_ds:
    probs = best_model.predict(images, verbose=0)
    y_true.extend(labels.numpy().tolist())
    y_pred.extend(np.argmax(probs, axis=1).tolist())

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, xticks_rotation=45, cmap='Blues')
plt.title('Ma trận nhầm lẫn')
plt.tight_layout()
plt.show()

hist = merge_history(history_head, history_fine)
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(hist['accuracy'], label='train')
plt.plot(hist['val_accuracy'], label='valid')
plt.title('Độ chính xác')
plt.xlabel('Epoch')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(hist['loss'], label='train')
plt.plot(hist['val_loss'], label='valid')
plt.title('Loss')
plt.xlabel('Epoch')
plt.legend()
plt.tight_layout()
plt.show()

final_model_path = MODEL_DIR / 'face_emotion_mobilenetv2_v2.keras'
best_model.save(final_model_path)

metadata = {
    'class_names': CLASS_NAMES,
    'img_size': IMG_SIZE,
    'kaggle_datasets': KAGGLE_DATASETS,
    'dataset_roots': [str(p) for p in dataset_roots],
    'train_samples': int(len(train_df)),
    'valid_samples': int(len(valid_df)),
    'test_samples': int(len(test_df)),
    'test_accuracy': float(test_acc),
    'class_weights': class_weights,
}

with open(MODEL_DIR / 'metadata_v2.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print('Đã lưu model:', final_model_path)
print('Đã lưu metadata:', MODEL_DIR / 'metadata_v2.json')

## 10. Dự đoán một ảnh và detect khuôn mặt

In [ ]:
def predict_emotion(image_path, model=best_model):
    image = Image.open(image_path).convert('RGB').resize(IMG_SIZE)
    array = np.asarray(image, dtype=np.float32)[None, ...]
    probs = model.predict(array, verbose=0)[0]
    pred_idx = int(np.argmax(probs))

    plt.figure(figsize=(4, 4))
    plt.imshow(image)
    plt.axis('off')
    plt.title(f'{CLASS_NAMES[pred_idx]} - {probs[pred_idx]:.2%}')
    plt.show()

    return pd.DataFrame({'emotion': CLASS_NAMES, 'probability': probs}).sort_values('probability', ascending=False)


face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')


def detect_faces_and_predict(image_path, model=best_model):
    bgr = cv2.imread(str(image_path))
    if bgr is None:
        raise FileNotFoundError(image_path)

    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40))

    for (x, y, w, h) in faces:
        face = bgr[y:y+h, x:x+w]
        face_rgb = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
        face_rgb = cv2.resize(face_rgb, IMG_SIZE)
        probs = model.predict(face_rgb[None, ...].astype(np.float32), verbose=0)[0]
        pred = CLASS_NAMES[int(np.argmax(probs))]
        conf = float(np.max(probs))

        cv2.rectangle(bgr, (x, y), (x+w, y+h), (0, 255, 0), 2)
        cv2.putText(bgr, f'{pred}: {conf:.2f}', (x, max(y-10, 20)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(8, 6))
    plt.imshow(rgb)
    plt.axis('off')
    plt.show()

    return faces

# Ví dụ:
# predict_emotion('path_to_image.jpg')
# detect_faces_and_predict('path_to_face_image.jpg')